# HiGLUE Diagonal Integration of Human PFC Data

In this tutorial we outline the full HiGLUE pipeline to integrate snm3C-seq single-cell contact maps in human prefrontal cortex with scRNA-seq expression profiles from independent cells:

* snm3C-seq Human PFC: https://www.nature.com/articles/s41592-019-0547-z

* scRNA-seq Human PFC: https://autism.cells.ucsc.edu/

The scRNA-seq data should be in AnnData format and must contain a `counts` layer with the raw gene count matrix and a `celltype` obs column (note that it can just be a single arbitrary value representing "unknown")

In [6]:
import scanpy as sc

rna = sc.read('/mnt/jinstore/JinLab01/dmp131/scRNA/brain_autism_PFC_neurons.h5ad')

In [7]:
rna.layers['counts']

<12531x29738 sparse matrix of type '<class 'numpy.float32'>'
	with 45673533 stored elements in Compressed Sparse Row format>

In [8]:
rna.obs['celltype'].value_counts()

L2/3     3701
L4       2245
Sst      1414
Vip      1357
L5       1253
Pvalb    1123
L6        996
Ndnf      442
Name: celltype, dtype: int64

The scHi-C data is input the exact same as our SCORE package (https://github.com/JinLabBioinfo/SCORE) which documents how to obtain a .scool file and a metadata reference file.

To run HiGLUE, we provide the scRNA-seq as an AnnData object and a corresponding GTF file, the scHi-C data as a .scool file and metadata reference file, and several other arguments for preprocessing and training. The arguments are divided in half between HiGLUE specific arguments at the start, and scHi-C specific preprocessing arguments which are passed to the SCORE package when loading the scHi-C dataset. 

There are two main functionalities in the `higlue.py` script:

* `--preprocess` will load the scRNA and scHi-C data, map them to appropriate genomic coordinates, and construct the guidance graph for linking the features.

* `--train` will look for the preprocessed scRNA, scHi-C, and guidance graph, and then train the HiGLUE model

Note that these can be run simultaneously, but training is only possible if the preprocessed inputs are present. Some important arguments are:

* `--n_strata` indicates the max interaction distance considered (same usage as in `SCORE`)

* `--n_genes` number of highly-variable genes to use when embedding scRNA and constructing guidance graph

* `--min_count` interactions with fewer total counts across the dataset are dropped

* `--gene_list` provide extra genes which will always be included in guidance graph even if they are not in the set of HVGs

* `--loop_q` quantile cutoff for including a pseudo-bulk interaction in the guidance graph

In [12]:
! python ../higlue.py --rna_file /mnt/jinstore/JinLab01/dmp131/scRNA/brain_autism_PFC_neurons.h5ad \
						--gtf /mnt/jinstore/JinLab01/dmp131/scRNA/hg19_genes.gtf \
						--use_2d \
						--offset 1 \
						--n_strata 10 \
						--n_genes 5000 \
						--min_count 3 \
						--gene_list NDNF SV2C GAD1 \
						--loop_q 0.99 \
						--preprocess \
						--train \
						SCORE \
						--dset pfc_neurons \
						--scool /mnt/jinstore/JinLab02/dmp131/HiGLUE/scools/pfc_all_dist_200kb.scool \
						--reference /mnt/jinstore/JinLab02/dmp131/HiGLUE/data/pfc_neurons_ref \
						--resolution 200kb \
						--min_depth 50000;

Parsing SCORE args...
['--dset', 'pfc_neurons', '--scool', '/mnt/jinstore/JinLab02/dmp131/HiGLUE/scools/pfc_all_dist_200kb.scool', '--reference', '/mnt/jinstore/JinLab02/dmp131/HiGLUE/data/pfc_neurons_ref', '--resolution', '200kb', '--min_depth', '50000']
Dataset: pfc
Total Cells: 586
Cells before filtering: 586
Cells before filtering by chr: 586
reference depth < min_depth : 1026
chr_reads < chr_length : 24
not filtered : 562
Reference file with filtering criteria saved to data/pfc_filtered_ref
L2/3: 172
L5: 51
Vip: 73
Ndnf: 51
L6: 32
L4: 49
Pvalb: 45
Sst: 89
Filtering top loops in each chromosome...
100%|███████████████████████████████████████████| 24/24 [00:08<00:00,  2.83it/s]
       bin1_id  bin2_id  count      rank
0            3        4   1137  0.780259
1            4        5   3575  0.951263
2            4        6   1357  0.817427
3            4        7    709  0.654136
4            5        6   2669  0.917401
...        ...      ...    ...       ...
50465    14059    14060